# Imputation Technique 1: Dropping High-Missing Columns

**Dataset:** `Loan_Default.csv`

**When to use:** When a column has so many missing values that imputing them would introduce too much noise or distortion.

**Key concept:** If a column is missing more than a threshold (the lab uses **> 10,000** missing values), it's more statistically honest to drop the whole column rather than fabricate too much data.

---


### Step 1: Setup & Data Loading


In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# Load data
df = pd.read_csv('../../data/raw/Loan_Default.csv')
df.drop(['ID', 'year'], axis=1, inplace=True)

# Pre-encode features so we have a clean numerical/categorical baseline
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

enc = OrdinalEncoder()
df[Ordinal_features] = enc.fit_transform(df[Ordinal_features])

df_prep = df.copy()
for c in Nominal_features:
    df_prep[c + '_freq'] = df_prep[c].map(df_prep.groupby(c).size() / df_prep.shape[0])
    indexer = pd.factorize(df_prep[c], sort=True)[1]
    df_prep[c] = indexer.get_indexer(df_prep[c])
df_prep = df_prep.drop(Nominal_features, axis=1)

print(f'Starting shape: {df_prep.shape}')

Starting shape: (148670, 32)


### Step 2: Identify Missing Values Per Column

Before deciding what to drop, we inspect the scale of missing data.


In [2]:
missing = df_prep.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

Upfront_charges                   39642
Interest_rate_spread              36639
rate_of_interest                  36439
dtir1                             24121
LTV                               15098
property_value                    15098
income                             9150
loan_limit_freq                    3344
approv_in_adv_freq                  908
age                                 200
submission_of_application_freq      200
loan_purpose_freq                   134
Neg_ammortization_freq              121
term                                 41
dtype: int64


### Step 3: Drop High-Missing Columns

The lab defines "high missing" as > 10,000 missing values. These columns are dropped entirely.


In [3]:
high_missing_cols = [
    'rate_of_interest',
    'Interest_rate_spread',
    'Upfront_charges',
    'property_value',
    'LTV',
    'dtir1'
]

df_drop = df_prep.drop(high_missing_cols, axis=1)

print(f'Shape after dropping high-missing columns: {df_drop.shape}')
df_drop.head()

Shape after dropping high-missing columns: (148670, 26)


,loan_amount,term,income,Credit_Score,age,Status,loan_limit_freq,Gender_freq,approv_in_adv_freq,loan_type_freq,...,lump_sum_payment_freq,construction_type_freq,occupancy_type_freq,Secured_by_freq,total_units_freq,credit_type_freq,co-applicant_credit_type_freq,submission_of_application_freq,Region_freq,Security_Type_freq
0,116500,360.0,1740.0,758,0.0,1,0.910392,0.253306,0.838239,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.277924,0.500383,0.644474,0.430591,0.999778
1,206500,360.0,4980.0,552,3.0,1,0.910392,0.284832,0.838239,0.139652,...,0.022762,0.999778,0.929582,0.999778,0.985269,0.102899,0.499617,0.644474,0.502603,0.999778
2,406500,360.0,9480.0,834,1.0,0,0.910392,0.284832,0.155653,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.277924,0.500383,0.644474,0.430591,0.999778
3,456500,360.0,11880.0,587,2.0,0,0.910392,0.284832,0.838239,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.277924,0.500383,0.354180,0.502603,0.999778
4,696500,360.0,10440.0,602,0.0,0,0.910392,0.278462,0.155653,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.295292,0.499617,0.354180,0.502603,0.999778


### Step 4: Verify Remaining Missing Values

After dropping, we check what missing data remains. These will be handled by other strategies like Mode or Mean imputation.


In [4]:
remaining_missing = df_drop.isna().sum()
print(remaining_missing[remaining_missing > 0])

term                                41
income                            9150
age                                200
loan_limit_freq                   3344
approv_in_adv_freq                 908
loan_purpose_freq                  134
Neg_ammortization_freq             121
submission_of_application_freq     200
dtype: int64
